# Providence Tower Embedding Export (Colab)

This notebook focuses only on embedding and JSON export.

Flow:
1. Read `*.chunked.md` files
2. Parse chunk metadata and text
3. Filter short chunks
4. Embed in batches (GPU when available)
5. Export embeddings as JSONL parts + `manifest.json`

No Redis operations are performed in this notebook. Optimized for Google Colab server.

In [ ]:
# torch cleaner for some occasion
!pip uninstall -y torchcodec torch torchvision torchaudio numpy

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from __future__ import annotations

import json
import re
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer

In [ ]:
print(f"NumPy version: {np.__version__}")
print(f"Torch version: {torch.__version__}")
print("Sentence Transformers loaded!")

NumPy version: 2.0.2
Torch version: 2.10.0+cu128
Sentence Transformers loaded!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Put your chunked markdown files under MyDrive, e.g.:
# /content/drive/MyDrive/providencetower/chunked_md

Mounted at /content/drive


In [ ]:
# ---- Config ----
INPUT_DIR = "/content/drive/MyDrive/Work/Providence Tower v2/YsWikiChunks"  # change this path
MODEL_NAME = "BAAI/bge-small-en-v1.5"
BATCH_SIZE = 256
NORMALIZE_EMBEDDINGS = True
MIN_CHARS = 60

# JSON export output
EXPORT_DIR = "/content/drive/MyDrive/Work/Providence Tower v2/embedding_export_json"
WRITE_PART_SIZE = 256  # rows per JSONL part

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


In [ ]:
@dataclass(frozen=True)
class ChunkDocument:
    chunk_id: str
    page_id: int
    page_title: str
    section: str
    subsection: str
    source_file: str
    text: str


class EmbeddingService:
    def __init__(
        self,
        *,
        model_name: str,
        device: str,
        batch_size: int,
        normalize_embeddings: bool = True,
    ) -> None:
        self.model_name = model_name
        self.batch_size = batch_size
        self.normalize_embeddings = normalize_embeddings

        if device == "cpu":
            torch.set_num_threads(4)

        self.model = SentenceTransformer(model_name, device=device)
        self._breadcrumb_pattern = re.compile(
            r"^\[Page:\s*(.*?)\]\[Section:\s*(.*?)\]\[Subsection:\s*(.*?)\]\s*(.*)$",
            re.DOTALL,
        )
        self._meta_from_filename_pattern = re.compile(r"^(?P<id>\d+)__(?P<title>.+)\.chunked\.md$", re.IGNORECASE)

    def _extract_page_metadata_from_file(self, filename: str) -> tuple[int, str]:
        match = self._meta_from_filename_pattern.match(filename)
        if not match:
            return -1, filename
        page_id = int(match.group("id"))
        page_title = match.group("title").replace("_", " " ).strip()
        return page_id, page_title

    def _should_embed_document(self, document: ChunkDocument) -> bool:
        return len(document.text.strip()) >= MIN_CHARS

    def load_documents_from_file(self, file_path: Path) -> list[ChunkDocument]:
        text = file_path.read_text(encoding="utf-8")
        fallback_page_id, fallback_title = self._extract_page_metadata_from_file(file_path.name)
        docs: list[ChunkDocument] = []

        for chunk_block in re.split(r"(?m)^##\s+Chunk\s+", text)[1:]:
            chunk_header, _, body = chunk_block.partition("\n")
            chunk_id = chunk_header.strip()
            body = body.strip()
            if not body:
                continue

            page_id = fallback_page_id
            page_title = fallback_title
            section = "General"
            subsection = "General"
            chunk_text = body

            breadcrumb_match = self._breadcrumb_pattern.match(body)
            if breadcrumb_match:
                page_title = breadcrumb_match.group(1).strip() or fallback_title
                section = breadcrumb_match.group(2).strip() or "General"
                subsection = breadcrumb_match.group(3).strip() or "General"
                chunk_text = breadcrumb_match.group(4).strip()

            if not chunk_text:
                continue

            docs.append(
                ChunkDocument(
                    chunk_id=chunk_id,
                    page_id=page_id,
                    page_title=page_title,
                    section=section,
                    subsection=subsection,
                    source_file=file_path.name,
                    text=chunk_text,
                )
            )

        return docs

    def load_documents_from_directory(self, input_dir: Path) -> list[ChunkDocument]:
        if not input_dir.exists():
            raise FileNotFoundError(f"Chunk input directory not found: {input_dir}")

        files = sorted(input_dir.glob("*.chunked.md"))
        documents: list[ChunkDocument] = []
        for file_path in files:
            documents.extend(self.load_documents_from_file(file_path))
        return documents

    def filter_documents_for_embedding(self, documents: list[ChunkDocument]) -> list[ChunkDocument]:
        return [document for document in documents if self._should_embed_document(document)]

    def iter_embeddings_by_batch(self, documents: list[ChunkDocument], *, batch_size: int | None = None):
        eligible_documents = self.filter_documents_for_embedding(documents)
        if not eligible_documents:
            return

        effective_batch_size = batch_size or self.batch_size
        if effective_batch_size < 1:
            raise ValueError("batch_size must be >= 1")

        for start in range(0, len(eligible_documents), effective_batch_size):
            batch_docs = eligible_documents[start:start + effective_batch_size]
            texts_to_embed = [
                f"[Page: {doc.page_title}][Section: {doc.section}][Subsection: {doc.subsection}] {doc.text}"
                for doc in batch_docs
            ]
            batch_vectors = self.model.encode(
                texts_to_embed,
                batch_size=self.batch_size,
                show_progress_bar=False,
                convert_to_numpy=True,
                normalize_embeddings=self.normalize_embeddings,
            )
            yield batch_docs, np.asarray(batch_vectors, dtype=np.float32)

In [ ]:
# Embedding-only run: export JSONL parts + manifest.json
start = time.perf_counter()

embedding_service = EmbeddingService(
    model_name=MODEL_NAME,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    normalize_embeddings=NORMALIZE_EMBEDDINGS,
)

input_dir = Path(INPUT_DIR)
export_dir = Path(EXPORT_DIR)
export_dir.mkdir(parents=True, exist_ok=True)

print(f"Loading chunk documents from: {input_dir}")
documents = embedding_service.load_documents_from_directory(input_dir)
if not documents:
    raise RuntimeError(f"No chunk documents found in: {input_dir}")

filtered_documents = embedding_service.filter_documents_for_embedding(documents)
skipped_documents = len(documents) - len(filtered_documents)
if skipped_documents:
    print(f"Skipping {skipped_documents} chunks shorter than {MIN_CHARS} characters")
if not filtered_documents:
    raise RuntimeError("No chunk documents met the minimum length requirement")

print(f"Embedding and exporting {len(filtered_documents)} chunks using model '{MODEL_NAME}'")

manifest = {
    "status": "ok",
    "mode": "embedding-json-export",
    "model_name": MODEL_NAME,
    "normalize_embeddings": NORMALIZE_EMBEDDINGS,
    "input_dir": str(input_dir),
    "export_dir": str(export_dir),
    "total_documents": len(documents),
    "eligible_documents": len(filtered_documents),
    "skipped_documents": skipped_documents,
    "vector_dim": None,
    "parts": [],
}

written = 0
batch_index = 0

for batch_docs, batch_vectors in embedding_service.iter_embeddings_by_batch(documents, batch_size=BATCH_SIZE):
    if batch_vectors.size == 0:
        continue

    if manifest["vector_dim"] is None:
        manifest["vector_dim"] = int(batch_vectors.shape[1])

    rows = []
    for doc, vec in zip(batch_docs, batch_vectors):
        rows.append(
            {
                "chunk_id": doc.chunk_id,
                "page_id": doc.page_id,
                "page_title": doc.page_title,
                "section": doc.section,
                "subsection": doc.subsection,
                "source_file": doc.source_file,
                "text": doc.text,
                "embedding": np.asarray(vec, dtype=np.float32).tolist(),
            }
        )

    for start_idx in range(0, len(rows), WRITE_PART_SIZE):
        batch_index += 1
        part_rows = rows[start_idx : start_idx + WRITE_PART_SIZE]
        part_file = export_dir / f"part_{batch_index:05d}.jsonl"
        with part_file.open("w", encoding="utf-8") as f:
            for row in part_rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

        written += len(part_rows)
        manifest["parts"].append(
            {
                "part": batch_index,
                "file": part_file.name,
                "rows": len(part_rows),
            }
        )
        print(f"Exported part {batch_index}: {len(part_rows)} rows (cumulative={written}/{len(filtered_documents)})")

if manifest["vector_dim"] is None:
    raise RuntimeError("No vectors generated from eligible documents")

manifest["written_rows"] = written
manifest_path = export_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote manifest: {manifest_path}")

elapsed = time.perf_counter() - start
summary = {
    "status": "ok",
    "mode": "embedding-json-export",
    "documents": len(filtered_documents),
    "rows_processed": written,
    "vector_dim": manifest["vector_dim"],
    "model_name": MODEL_NAME,
    "export_dir": str(export_dir),
    "manifest": str(manifest_path),
    "elapsed_seconds": round(elapsed, 2),
}
print(summary)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading chunk documents from: /content/drive/MyDrive/Work/Providence Tower v2/YsWikiChunks
Skipping 10281 chunks shorter than 60 characters
Embedding and exporting 14044 chunks using model 'BAAI/bge-small-en-v1.5'
Exported part 1: 256 rows (cumulative=256/14044)
Exported part 2: 256 rows (cumulative=512/14044)
Exported part 3: 256 rows (cumulative=768/14044)
Exported part 4: 256 rows (cumulative=1024/14044)
Exported part 5: 256 rows (cumulative=1280/14044)
Exported part 6: 256 rows (cumulative=1536/14044)
Exported part 7: 256 rows (cumulative=1792/14044)
Exported part 8: 256 rows (cumulative=2048/14044)
Exported part 9: 256 rows (cumulative=2304/14044)
Exported part 10: 256 rows (cumulative=2560/14044)
Exported part 11: 256 rows (cumulative=2816/14044)
Exported part 12: 256 rows (cumulative=3072/14044)
Exported part 13: 256 rows (cumulative=3328/14044)
Exported part 14: 256 rows (cumulative=3584/14044)
Exported part 15: 256 rows (cumulative=3840/14044)
Exported part 16: 256 rows (cumul

In [ ]:
# Quick export sanity-check
export_dir = Path(EXPORT_DIR)
manifest_path = export_dir / "manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(f"Manifest not found: {manifest_path}. Run the export cell first.")

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print({
    "model_name": manifest.get("model_name"),
    "vector_dim": manifest.get("vector_dim"),
    "parts": len(manifest.get("parts", [])),
    "written_rows": manifest.get("written_rows"),
})

if manifest.get("parts"):
    sample_part = export_dir / manifest["parts"][0]["file"]
    print(f"Sample part file: {sample_part}")
    with sample_part.open("r", encoding="utf-8") as f:
        first_line = f.readline().strip()
    print("First row preview:")
    print(first_line[:600] + ("..." if len(first_line) > 600 else ""))

{'model_name': 'BAAI/bge-small-en-v1.5', 'vector_dim': 384, 'parts': 55, 'written_rows': 14044}
Sample part file: /content/drive/MyDrive/Work/Providence Tower v2/embedding_export_json/part_00001.jsonl
First row preview:
{"chunk_id": "10033-0003", "page_id": 10033, "page_title": "Luta Gemma", "section": "Basic info", "subsection": "Alternative spellings", "source_file": "10033__Luta_Gemma.chunked.md", "text": "*Luther Gemma* (Ys I OVA, subtitles)\n*Jemma* (Ys I OVA, JP credits)\n*Ruta* (Ys I OVA, EN credits)", "embedding": [-0.03311018645763397, -0.07490409910678864, 0.023290082812309265, 0.017604218795895576, 0.006089846137911081, 0.004370860289782286, -0.021820958703756332, 0.03178468346595764, 0.018084567040205002, -0.02626017853617668, 0.024044744670391083, -0.052712295204401016, -0.014730923809111118, 0.0...
